4.2 Extension 1 (Methodological): Semantic Supervision (2.5 Points)
This extension addresses the methodological critique raised in section 2.2: the weakness of Fuzzy Matching.
• The Concept: Instead of training SigExt on labels generated via simple character overlap, we will generate labels based on semantic similarity.
• Implementation:
• We will use a Sentence-BERT model (e.g., all-MiniLM-L6-v2 or paraphrase-multilingual-mpnet-base-v2) to encode all sentences of the source document (P_src) and all sentences of the target summary (P_ref) into dense vectors (embeddings).
• We will calculate the Cosine Similarity between the vectors.
• We will define a new labeling function:
• $$\\text{Label}(p) = 1 \\iff \\max\_{q \\in P\_{ref}} \\text{CosSim}(\\vec{p}, \\vec{q}) > \\theta$$
• where
theta is an experimental threshold (e.g., 0.6).
• Scientific Value: This modification transforms the extraction paradigm from "lexical" to "semantic." The hypothesis is that SigExt will learn to extract concepts, not just words, improving the quality of the signal sent to the LLM. This is an original scientific contribution that elevates the project above simple reproduction.


In [5]:
#this is the dataset we are going to use for the first test: https://huggingface.co/datasets/ccdv/arxiv-summarization
!pip install datasets sentence-transformers nltk scikit-learn

from datasets import load_dataset

dataset = load_dataset(
    "ccdv/arxiv-summarization",
    "section",
    split="train"
)


README.md: 0.00B [00:00, ?B/s]

section/train-00000-of-00015.parquet:   0%|          | 0.00/230M [00:00<?, ?B/s]

section/train-00001-of-00015.parquet:   0%|          | 0.00/228M [00:00<?, ?B/s]

section/train-00002-of-00015.parquet:   0%|          | 0.00/228M [00:00<?, ?B/s]

section/train-00003-of-00015.parquet:   0%|          | 0.00/227M [00:00<?, ?B/s]

...

section/validation-00000-of-00001.parque(…):   0%|          | 0.00/105M [00:00<?, ?B/s]

section/test-00000-of-00001.parquet:   0%|          | 0.00/105M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/203037 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/6436 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/6440 [00:00<?, ? examples/s]

In [6]:
from sentence_transformers import SentenceTransformer
embedder = SentenceTransformer("sentence-transformers/all-mpnet-base-v2")
from sklearn.metrics.pairwise import cosine_similarity

CONFIG = {
    'BERT_MODEL': 'all-MiniLM-L6-v2',
    'THRESHOLD': 0.6
}



In [7]:
sentences = [
    "The company declared bankruptcy",
    "the firm went out of business",
    "my dog is dori"

]

embs = embedder.encode(sentences)
cosine_similarity(embs)



array([[ 1.        ,  0.78523743,  0.00689894],
       [ 0.78523743,  1.0000001 , -0.03752092],
       [ 0.00689894, -0.03752092,  1.        ]], dtype=float32)

In [ ]:
dataset

In [8]:
samples = dataset[0: 100]

In [9]:
articles = samples["article"]
abstract = samples["abstract"]

from transformers import AutoTokenizer
CONFIG["LONGFORMER_MODEL"] = "markussagen/xlm-roberta-longformer-base-4096"

tokenizer = AutoTokenizer.from_pretrained(CONFIG["LONGFORMER_MODEL"])

tokenizer_config.json:   0%|          | 0.00/212 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/773 [00:00<?, ?B/s]

sentencepiece.bpe.model:   0%|          | 0.00/5.07M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/150 [00:00<?, ?B/s]

In [10]:
import nltk
nltk.download("punkt_tab")
from nltk.tokenize import sent_tokenize

article_sentences = []
for article_text in articles:
    article_sentences.extend(sent_tokenize(article_text))

abstract_sentences = []
for abstract_text in abstract:
    abstract_sentences.extend(sent_tokenize(abstract_text))


[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt_tab.zip.


In [11]:
embs_article = embedder.encode(article_sentences)
embs_abstract = embedder.encode(abstract_sentences)


In [16]:
import numpy as np

embs_articles = np.array(embs_article)
embs_abstracts = np.array(embs_abstract)

def normalize(v):
  return v / np.linalg.norm(v, axis=1, keepdims=True)

embs_article = normalize(embs_article)
embs_abstracts = normalize(embs_abstracts)

cosine_sim = embs_articles @ embs_abstracts.T
print(cosine_sim)

[[ 0.86523986  0.58252573  0.47581416 ... -0.10677973 -0.00925716
  -0.06471001]
 [ 0.706417    0.37901175  0.26708    ... -0.11948358 -0.02951917
  -0.06754551]
 [ 0.7751018   0.53451455  0.49191955 ... -0.03944216  0.01384743
  -0.00672613]
 ...
 [-0.02357909 -0.0196852   0.01981504 ...  0.6925951   0.4277715
   0.38523075]
 [-0.02747959 -0.01175584 -0.04967383 ...  0.0469471   0.00663769
  -0.02635331]
 [ 0.06945735  0.19437611  0.14261577 ...  0.6053724   0.35382906
   0.28443548]]
